# 04 — Isolation Forest Training

This notebook trains an Isolation Forest model for unsupervised anomaly detection:
- Load preprocessed features (no labels needed for training)
- Train the model on the full dataset
- Analyze anomaly scores and decision threshold
- Evaluate detection performance against known labels
- Persist the trained model to disk

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

sys.path.insert(0, os.path.join(os.path.dirname("__file__" if "__file__" in dir() else ""), ".."))
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

## 1. Load Preprocessed Data

In [ ]:
FEATURES_PATH = os.path.join("..", "data", "processed", "features_scaled.npy")
LABELS_PATH = os.path.join("..", "data", "processed", "labels.npy")
META_PATH = os.path.join("..", "data", "processed", "feature_columns.npy")

if os.path.exists(FEATURES_PATH):
    X = np.load(FEATURES_PATH)
    y_true = np.load(LABELS_PATH)
    feature_names = np.load(META_PATH, allow_pickle=True).tolist()
else:
    # Fallback: build from raw CSV inline
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    RAW_PATH = os.path.join("..", "data", "raw", "CICIDS2017", "Wednesday-workingHours.pcap_ISCX.csv")
    df = pd.read_csv(RAW_PATH).drop_duplicates().reset_index(drop=True)
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["Label"]).reset_index(drop=True)
    META_COLS = ["Flow ID", "Source IP", "Source Port", "Destination IP",
                 "Destination Port", "Protocol", "Timestamp", "Label"]
    feature_names = [c for c in df.columns if c not in META_COLS and df[c].dtype in ["float64", "int64"]]
    X_raw = df[feature_names].replace([np.inf, -np.inf], np.nan).fillna(0).values
    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)
    le = LabelEncoder()
    y_true = le.fit_transform(df["Label"])

print(f"Feature matrix: {X.shape}")
print(f"Labels:          {y_true.shape}")
print(f"Unique labels:   {np.unique(y_true)}")

## 2. Train Isolation Forest

In [ ]:
model = IsolationForest(
    n_estimators=100,
    contamination=0.1,
    max_samples="auto",
    random_state=42,
    n_jobs=-1,
)

model.fit(X)
print(f"Model trained: {model.n_estimators} estimators, contamination={model.contamination}")

## 3. Anomaly Scores

In [ ]:
scores = model.decision_function(X)
predictions = model.predict(X)  # -1 = anomaly, 1 = normal

print(f"Score range: [{scores.min():.4f}, {scores.max():.4f}]")
print(f"Anomalies detected: {(predictions == -1).sum():,} / {len(predictions):,}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(scores, bins=80, edgecolor="black", alpha=0.7)
axes[0].axvline(x=0, color="red", linestyle="--", label="Decision boundary")
axes[0].set_title("Isolation Forest Anomaly Scores")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Count")
axes[0].legend()

# Scores colored by true label (binary: 0=BENIGN, other=ATTACK)
is_attack = y_true != 0
axes[1].hist(scores[~is_attack], bins=80, alpha=0.6, label="BENIGN", color="steelblue")
axes[1].hist(scores[is_attack], bins=80, alpha=0.6, label="ATTACK", color="tomato")
axes[1].set_title("Score Distribution by True Label")
axes[1].set_xlabel("Score")
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Threshold Analysis

In [ ]:
# Binary ground truth: 0 = BENIGN, 1 = ATTACK
y_binary = (y_true != 0).astype(int)

thresholds = np.linspace(scores.min(), scores.max(), 20)
results = []

for t in thresholds:
    preds = (scores < t).astype(int)
    results.append({
        "threshold": t,
        "accuracy": accuracy_score(y_binary, preds),
        "precision": precision_score(y_binary, preds, zero_division=0),
        "recall": recall_score(y_binary, preds, zero_division=0),
        "f1": f1_score(y_binary, preds, zero_division=0),
    })

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
for metric in ["accuracy", "precision", "recall", "f1"]:
    ax.plot(res_df["threshold"], res_df[metric], label=metric)
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Metrics vs Anomaly Threshold")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Model Metrics

In [ ]:
# Use sklearn default threshold (0) mapped to binary
y_pred_binary = (predictions == -1).astype(int)

print("=" * 50)
print("ISOLATION FOREST — DEFAULT THRESHOLD (0)")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_binary, y_pred_binary):.4f}")
print(f"Precision: {precision_score(y_binary, y_pred_binary, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_binary, y_pred_binary, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_binary, y_pred_binary, zero_division=0):.4f}")
print("=" * 50)

cm = confusion_matrix(y_binary, y_pred_binary)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — Isolation Forest")
plt.tight_layout()
plt.show()

## 6. Save Model

In [ ]:
MODEL_DIR = os.path.join("..", "models", "trained")
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, "isolation_forest.pkl")
joblib.dump(model, model_path)
print(f"Model saved to: {os.path.abspath(model_path)}")
print(f"File size: {os.path.getsize(model_path) / 1024:.1f} KB")